In [ ]:
# HyperOpt 를 찾기 위한 기본 루틴 정리본 ,
# 본 파일을 save_as 이니셜_1.모델축약어_FindHO.ipynb 로 저장하세요~ ex) lkj_1.XGB_FindHO.ipynb

In [1]:
import sys
import os

project_root = 'c:/big20/git/big20-ML-project2-team3/CreditCardFraud'

# sys.path에 추가 (모듈 import용)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [2]:
import pandas as pd
import numpy  as np
from scipy.special import logit
import time
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path
import math
import shap

from xgboost import XGBClassifier, plot_importance
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split, KFold, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, roc_curve, classification_report, confusion_matrix, recall_score, precision_score
from sklearn.datasets import make_classification
from sklearn.pipeline import make_pipeline
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials

import matplotlib
import matplotlib.pyplot as plt
plt.rcParams['font.family'] ='Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False
import seaborn as sns

import importlib
from utils import preprocessing
importlib.reload(preprocessing) # 모듈 reload
# importlib.reload(user_utils)

import utils.preprocessing as pp
import utils.user_utils    as uu
import utils.model_utils   as mu
import utils.evaluation    as ev
import utils.modeling      as mo
import utils.data_sampling as ds

# train = pd.read_csv("../data/train.csv")
# test  = pd.read_csv("../data/test.csv")

In [3]:
# 결과받을 딕셔너리
results = {}
team_rs = 23 # 우리팀 random_state

In [4]:
#1. 데이터 로딩
raw_df = pp.ccf_load_data()

데이터 로드 성공: (284807, 31)


In [5]:
# 2. Time 컬럼 삭제 , 데이터,타겟 분리
X_features, y_target = pp.split_features_target(raw_df, cols= 'Time')

In [6]:
# 3. 이상치를 경계값으로 치환
cap_X_feature = pp.cap_outliers(X_features)

In [7]:
# 4.1 학습/테스트 데이터 분리
X_train, X_test, y_train, y_test = pp.data_split(cap_X_feature, y_target)

In [8]:
# 5.1 학습/검증 데이터 분리
X_tr, X_val, y_tr, y_val = pp.data_split(X_train, y_train, size=0.4)

In [9]:
# 4.2 Over Sampling 하는 경우
X_over, y_over = ds.oversampling_smote(X_train, y_train)

✅ SMOTE 오버샘플링 완료
   원본 샘플 수: 227845 (Class 0: 227451, Class 1: 394)
   샘플링 후: 454902 (Class 0: 227451, Class 1: 227451)


In [10]:
# 5.2 Over Sampling한 경우 학습/검증 데이터 분리
X_tr_over, X_val_over, y_tr_over, y_val_over = pp.data_split(X_over, y_over, size=0.4)

In [11]:
# 5-1. train/test 전체에 대해 스케일러 학습 & 변환
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# 5-2. HyperOpt용 train/val 분리 (스케일된 데이터 기준)
#     - y_tr, y_val도 다시 분리
X_tr_scaled, X_val_scaled, y_tr_scaled, y_val_scaled = pp.data_split(
    X_train_scaled, y_train, size=0.4
)

In [12]:
# =========================
# 1) Base / Meta 모델 정의
# GB + RF + XGB / LoG
# =========================

# 1-1. GradientBoosting (HyperOpt 결과 사용)
best_gb_params = {
    'learning_rate': 0.02083271293464231,
    'max_depth': 3,
    'n_estimators': 800,
    'subsample': 0.7076395576782778,
    'random_state': 23
}
gb_base = GradientBoostingClassifier(**best_gb_params)

# 1-2. RandomForest (기본 튜닝 예시 – 필요시 팀이 찾은 best params로 교체 가능)
rf_best_params = {
    'bootstrap': True,          # 부트스트래핑 사용 여부 (True 권장)
    'class_weight': 'balanced', # 클래스 불균형 자동 처리 ('balanced' 사용)
    'criterion': 'entropy',     # 불순도 측정 기준: 엔트로피 사용 ('gini' 대신)
    'max_depth': 10, #3,         # 트리의 최대 깊이 (매우 얕게 설정됨)
    'max_features': None,       # 각 분기에서 고려할 최대 특성 수 (모든 특성 사용)
    'min_samples_leaf': 9,      # 리프 노드가 되기 위한 최소 샘플 수
    'min_samples_split': 3,     # 노드를 분할하기 위한 최소 샘플 수
    'n_estimators': 200,        # 생성할 트리 개수
    'random_state': 23,         # 재현성을 위한 랜덤 시드
    'n_jobs': -1                # 병렬 처리 시 모든 CPU 코어 사용
}
rf_base = RandomForestClassifier(**rf_best_params)

# 1-3. XGBoost (팀에서 공유했던 stacking 예시 파라미터 기반)
# scale_pos_weight = 음성/양성 비율
# n_pos = (y_train == 1).sum()
# n_neg = (y_train == 0).sum()
# scale_pos_weight = n_neg / n_pos
xgb_best_params = {  # 전처리 후 org data로 한 경우
    "colsample_bytree": 1.0,
    "gamma": 1.7089112007254605,
    "learning_rate": 0.038583724829444874,
    "max_depth": 3,
    "min_child_weight": 6,
    "n_estimators": 150,
    "reg_alpha": 0.5940048544595263,
    "reg_lambda": 0.007186370173139192,
    "scale_pos_weight": 82.0,
    "subsample": 0.6,
    "random_state": 23,
}
xgb_base = XGBClassifier(**xgb_best_params)

# 1-4. Base estimators 리스트 (최소 3개 조건 충족)
estimators = [
    ("gb",  gb_base),
    ("rf",  rf_base),
    ("xgb", xgb_base),
]

# 1-5. Meta learner: LogisticRegression (SMOTE + HyperOpt best)
lr_best_params_smote = {
    'C': 99.78576192595759,
    'class_weight': 'balanced',
    'max_iter': 550,
    'penalty': 'l2',
    'solver': 'lbfgs',
    'random_state': 23,
    'n_jobs': -1
}
meta_lr = LogisticRegression(**lr_best_params_smote)

In [13]:
# =========================
# 2) StackingClassifier 구성
# =========================

stack_clf = StackingClassifier(
    estimators=estimators,
    final_estimator=meta_lr,
    stack_method="predict_proba",  # base 모델들의 확률출력 사용
    passthrough=False,             # True로 하면 원래 feature까지 meta에 같이 넣음 (원하면 바꿔도 됨)
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=team_rs),
    n_jobs=-1,
)

# 주의: 이미 X_train_scaled / X_test_scaled 로 스케일링 끝난 상태라
# 굳이 Pipeline 안에 StandardScaler를 또 넣을 필요는 없음.
# 그래도 일관성을 위해 make_pipeline으로 감싸두면 나중에 변경이 쉬움.
pipe_stack = make_pipeline(stack_clf)

In [14]:
# =========================
# 3) 학습 + 평가 + 저장
# =========================

stack_model_name = "STACK_gb_rf_xgb_metaLR"

start_time = time.time()
results[stack_model_name] = uu.get_model_train_eval(
    pipe_stack,
    stack_model_name,
    X_train_scaled,
    X_test_scaled,
    y_train,
    y_test,
    hyperopt_params={
        "gb":   best_gb_params,
        "meta_lr_smote": lr_best_params_smote,
        # rf / xgb는 지금 예시는 수동 파라미터라 필요하면 나중에 dict로 추가
    },
)
exec_time = time.time() - start_time

print(f"\n[Stacking] 실행 시간: {exec_time:.2f}초")
print(f"[Stacking] 결과 요약: {results[stack_model_name]}")


✓ 모델 저장 완료: ../models\STACK_gb_rf_xgb_metaLR.pkl
  파일 크기: 3.75 MB
folder = c:\big20\git\big20-ML-project2-team3\CreditCardFraud\results
{'AUC': 0.9724, '정확도': 0.9893, '정밀도': 0.1265, '재현율': 0.8878, 'F1': 0.2214 }
{'오차행렬':
[[56263   601]
 [   11    87]] }
실행 시간: 6135.226485967636
하이퍼파라미터: {'gb': {'learning_rate': 0.02083271293464231, 'max_depth': 3, 'n_estimators': 800, 'subsample': 0.7076395576782778, 'random_state': 23}, 'meta_lr_smote': {'C': 99.78576192595759, 'class_weight': 'balanced', 'max_iter': 550, 'penalty': 'l2', 'solver': 'lbfgs', 'random_state': 23, 'n_jobs': -1}}

[Stacking] 실행 시간: 6135.34초
[Stacking] 결과 요약: {'AUC': 0.9724, '정밀도': 0.1265, '재현율': 0.8878, 'F1': 0.2214}


In [ ]:
# ============================================
# thresholds별 튜닝 접근 (LogisticRegression용)
# ============================================

from sklearn.metrics import precision_recall_curve  # 위에서 안 했으면 한 번만 추가

def threshold_tuning_logreg(model, X_test, y_test, model_name="LogReg"):
    """
    - model: 학습 완료된 LogisticRegression (predict_proba 사용 가능)
    - X_test, y_test: 평가용 데이터
    - model_name: 표/그래프 제목용 라벨
    """
    # 1) 예측 확률
    stack_pred_proba = pipe_stack.predict_proba(X_test_scaled)[:,1]

    # 2) threshold 그리드 정의 (0.1 ~ 0.9)
    thresholds = np.arange(0.1, 0.91, 0.1)
    results_threshold = []

    for thr in thresholds:
        y_pred_thr = (y_pred_proba >= thr).astype(int)

        precision = precision_score(y_test, y_pred_thr, zero_division=0)
        recall    = recall_score(y_test, y_pred_thr, zero_division=0)
        f1        = f1_score(y_test, y_pred_thr, zero_division=0)
        # AUC는 threshold와 무관하니 한 번만 계산해도 되지만,
        # yjh 코드 스타일 맞춰서 그대로 넣어준다.
        auc       = roc_auc_score(y_test, y_pred_proba)

        results_threshold.append({
            "threshold": thr,
            "precision": precision,
            "recall":    recall,
            "f1":        f1,
            "auc":       auc
        })

    # 3) DataFrame으로 정리
    df_thr = pd.DataFrame(results_threshold)
    print(f"\n==== {model_name} threshold별 성능 ====")
    print(df_thr)

    # 4) Precision-Recall 곡선 시각화
    precisions, recalls, thr_pr = precision_recall_curve(y_test, y_pred_proba)

    plt.figure(figsize=(8, 5))
    plt.plot(thr_pr, precisions[:-1], "b--", label="Precision")
    plt.plot(thr_pr, recalls[:-1],    "g-",  label="Recall")
    plt.title(f"{model_name} - Threshold vs Precision/Recall")
    plt.xlabel("Threshold")
    plt.ylabel("Score")
    plt.legend()
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.show()

    return df_thr

In [ ]:
# 기본 LogisticRegression (LoR_ho_best)에 대해 threshold 튜닝
df_thr_log_base = threshold_tuning_logreg(
    model=best_logreg,          # 이미 HyperOpt로 튜닝된 LR
    X_test=X_test_scaled,       # StandardScaler 적용된 원본 test
    y_test=y_test,
    model_name="LoR_ho_best"
)

In [ ]:
# BestOpt 찾고 나서 스케일적용 버전 만들어서 모델링하기
# X_train_scaled, X_test_scaled, scaler = pp.scale_data(X_train, X_test)